In [ ]:
# FIND the correct Drive path automatically
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Try common Drive mount points
candidates = [
    '/content/drive/MyDrive',
    '/content/drive/Shared drives/MyDrive',
    '/content/drive/Shareddrives/MyDrive',
    '/content/drive',
]

BASE = None
for c in candidates:
    test = os.path.join(c, 'standup4ai')
    if os.path.exists(test):
        BASE = c
        print(f'Found Drive at: {c}')
        break

if BASE is None:
    # List what's in /content/drive
    print('Contents of /content/drive:')
    if os.path.exists('/content/drive'):
        for item in os.listdir('/content/drive'):
            print(f'  {item}')
    else:
        print('/content/drive does not exist!')

# Find audio directory
audio_dirs = [
    os.path.join(BASE, 'standup4ai', 'audio_1000') if BASE else None,
    os.path.join(BASE, 'standup4ai', 'audio') if BASE else None,
    os.path.join(BASE, 'audio_1000') if BASE else None,
    '/content/audio_1000',
]
AUDIO_DIR = None
for d in audio_dirs:
    if d and os.path.exists(d):
        AUDIO_DIR = d
        print(f'Audio found at: {d}')
        break

if AUDIO_DIR is None:
    print('ERROR: Could not find audio directory!')
else:
    files = os.listdir(AUDIO_DIR)
    print(f'Audio dir has {len(files)} files')
    print(f'Sample files: {files[:3]}')


In [ ]:
# Setup with found paths
BASE = '/content/drive/MyDrive'
AUDIO_DIR = BASE + '/standup4ai/audio_1000'
LABEL_DIR = BASE + '/standup4ai/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/standup4ai/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/standup4ai/wordlevel_checkpoint.json'

import os, json, time, gc
import numpy as np, pandas as pd
import torch
from transformers import AutoModel
import librosa

os.makedirs(OUT_DIR, exist_ok=True)

# Delete corrupt checkpoint
if os.path.exists(CKPT_FILE):
    os.remove(CKPT_FILE)
    print('Deleted old checkpoint')

done = set()

audio_map = {}
for f in os.listdir(AUDIO_DIR):
    if not f.endswith(('.m4a', '.wav', '.mp3')):
        continue
    base = f.rsplit('.', 1)[0]
    if ',' in base:
        base = base.split(',')[0]
    audio_map[base] = os.path.join(AUDIO_DIR, f)

label_map = {}
for f in os.listdir(LABEL_DIR):
    if f.endswith('.csv'):
        label_map[f.replace('.csv', '')] = os.path.join(LABEL_DIR, f)

overlap = sorted(set(audio_map.keys()) & set(label_map.keys()) - done)
print(f'Audio: {len(audio_map)} | Labels: {len(label_map)} | To process: {len(overlap)}')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    torch.cuda.empty_cache()
    gc.collect()
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
SR = 16000
print('WavLM ready!')


In [ ]:
def parse_timestamp(ts):
    ts = str(ts).strip().strip('[]')
    p = ts.split(',')
    return float(p[0]), float(p[1])

def extract_one_word(audio_segment):
    seg_len = len(audio_segment)
    if seg_len < 400:
        return np.zeros(768, dtype=np.float32)
    # Convert to tensor: (1, N)
    audio_t = torch.tensor(audio_segment, dtype=torch.float32).unsqueeze(0).to(device)
    # WavLM direct forward - no feature_extractor needed
    with torch.no_grad():
        out = wavlm(audio_t)
        emb = out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
    del audio_t, out
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return emb.astype(np.float32)

print('Extractor ready!')


In [ ]:
t0 = time.time()
failed = []
for i, vid in enumerate(overlap):
    out_file = OUT_DIR + '/' + vid + '_word_features.npy'
    if os.path.exists(out_file):
        print(f'{i+1}/{len(overlap)} {vid}: already exists, skip')
        continue

    try:
        df = pd.read_csv(label_map[vid])
        word_times = []
        for _, row in df.iterrows():
            try:
                t0_w, t1_w = parse_timestamp(row['timestamp'])
                word_times.append((t0_w, t1_w))
            except Exception:
                continue

        if not word_times:
            print(f'{i+1}/{len(overlap)} {vid}: no valid timestamps')
            failed.append(vid)
            continue

        y_full, sr = librosa.load(audio_map[vid], sr=SR, mono=True)
        n_samples = len(y_full)

        all_emb = []
        for j, (t0_w, t1_w) in enumerate(word_times):
            s = int(t0_w * SR)
            e = min(int(t1_w * SR), n_samples)
            if e <= s:
                all_emb.append(np.zeros(768, dtype=np.float32))
                continue
            seg = y_full[s:e]
            emb = extract_one_word(seg)
            all_emb.append(emb)

        feats = np.vstack(all_emb)
        np.save(out_file, feats)
        done.add(vid)

        del y_full, all_emb
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    except Exception as e:
        print(f'{i+1}/{len(overlap)} {vid}: ERROR {e}')
        failed.append(vid)
        continue

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    rem_min = (len(overlap) - i - 1) / rate * 60 if rate > 0 else 0
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape} | done={len(done)} | {rate:.0f}/hr | ETA={rem_min:.0f}min')

    if len(done) % 5 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)
print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')
if failed:
    print(f'Failed: {len(failed)}: {failed}')


In [ ]:
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(OUT_DIR + '/' + f)
    print(f'  {f}: {d.shape}')
